# Week 3 — Jupyter workflow

**Goal**: turn the Week 1 CLI logic into a reproducible, narrative notebook that any teammate can `uv sync` and `Restart & Run All`.

**Done when**:
- `Restart & Run All` succeeds top-to-bottom with no manual steps
- Exported HTML is readable standalone: `uv run jupyter nbconvert --to html notebooks/02_jupyter_workflow.ipynb`
- A teammate can clone the repo and reproduce every output

**Structure**: Goal → Data → Analysis → Conclusion.

## Setup

`%autoreload 2` re-imports `src/` modules on every cell execution, so edits to `spatial_notebooks/` are picked up without a kernel restart.

In [1]:
%load_ext autoreload
%autoreload 2
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / "src"))

In [2]:
from pathlib import Path

from spatial_notebooks.stats import add_density, load_csv, summarize

REPO_ROOT = Path.cwd().parent
DATA = REPO_ROOT / "data" / "samples" / "nagoya_wards.csv"
DATA

PosixPath('/Users/mac/Projects/Eukarya/Projects/spatial-notebooks/data/samples/nagoya_wards.csv')

## Data

Question this cell answers: *what does the raw input look like?*

In [3]:
df = load_csv(DATA)
df.head()

,ward,population,area_km2,lat,lon
0,Naka,99120,9.38,35.1681,136.9066
1,Higashi,87230,7.71,35.1819,136.9304
2,Kita,165340,17.53,35.2086,136.9161
3,Nishi,153210,17.94,35.1890,136.8771
4,Nakamura,137800,16.30,35.1706,136.8812


In [4]:
summarize(df)

Summary(rows=16, columns=5, numeric_columns=['population', 'area_km2', 'lat', 'lon'], missing_by_column={'ward': 0, 'population': 0, 'area_km2': 0, 'lat': 0, 'lon': 0})

## Analysis

Question: *which wards are the densest, and how large is the spread?*

In [5]:
ranked = (
    add_density(df)
    .sort_values("density", ascending=False)
    [["ward", "population", "area_km2", "density"]]
    .round(1)
)
ranked

,ward,population,area_km2,density
1,Higashi,87230,7.7,11313.9
0,Naka,99120,9.4,10567.2
13,Showa,108650,10.9,9931.4
5,Mizuho,106340,11.2,9477.7
2,Kita,165340,17.5,9431.8
14,Chikusa,164330,18.2,9039.1
3,Nishi,153210,17.9,8540.1
4,Nakamura,137800,16.3,8454.0
6,Atsuta,66780,8.2,8143.9
11,Meito,166850,21.5,7756.9


In [6]:
ranked["density"].describe().round(1)

count       16.0
mean      8073.4
std       2052.4
min       3144.2
25%       7261.6
50%       8299.0
75%       9443.3
max      11313.9
Name: density, dtype: float64

## Conclusion

- Highest-density ward is at the top of `ranked`; lowest is at the bottom.
- Spread (max/min) shows roughly an order-of-magnitude gap across Nagoya's 16 wards.
- Next step (Week 4): visualize this ranking with matplotlib and bring in a second dataset for cross-cutting analysis.

**Reproducibility checklist**

- [ ] `Kernel → Restart & Run All` succeeds
- [ ] `uv run jupyter nbconvert --to html notebooks/02_jupyter_workflow.ipynb`
- [ ] Clear outputs before commit unless a figure/table is the point
- [ ] Teammate can `uv sync` and rerun